In [7]:
import pandas as pd
import numpy as np
from openbb import obb

In [8]:
#download US treasury rates across 10 maturities (1 month through 30 years). 
#compute the covariance matrix between different maturities.

In [11]:
rate=(obb.fixedincome.government.treasury_rates(
    start_date='2020-01-01', 
    freq='D').to_df()
.dropna()
.div(100))

cov=rate.cov() #covariance yields matrix
print(cov)

          month_1   month_3   month_6    year_1    year_2    year_3    year_5  \
month_1  0.000495  0.000502  0.000496  0.000467  0.000403  0.000363  0.000315   
month_3  0.000502  0.000515  0.000512  0.000485  0.000422  0.000381  0.000331   
month_6  0.000496  0.000512  0.000514  0.000490  0.000430  0.000390  0.000338   
year_1   0.000467  0.000485  0.000490  0.000472  0.000417  0.000381  0.000331   
year_2   0.000403  0.000422  0.000430  0.000417  0.000375  0.000344  0.000300   
year_3   0.000363  0.000381  0.000390  0.000381  0.000344  0.000318  0.000278   
year_5   0.000315  0.000331  0.000338  0.000331  0.000300  0.000278  0.000245   
year_7   0.000288  0.000301  0.000308  0.000301  0.000273  0.000253  0.000224   
year_10  0.000264  0.000275  0.000280  0.000273  0.000247  0.000229  0.000203   
year_20  0.000245  0.000256  0.000260  0.000253  0.000229  0.000212  0.000189   
year_30  0.000219  0.000228  0.000230  0.000223  0.000200  0.000185  0.000164   

           year_7   year_10

In [15]:
#Decompose the eigenvectors
# NumPy to get the eigenvalues of the covariance matrix so we can decompose it into the principal components.

#This step calculates the eigenvalues and eigenvectors of the covariance matrix where eigenvalues represent 
#the variance explained by each eigenvector. The square root of the eigenvalues is used to construct a diagonal matrix 
#which represents the standard deviation (volatility) associated with each principal component.

eigenvalues, eigenvectors = np.linalg.eig(cov)
lambda_sqrt = np.sqrt(eigenvalues)
eigv_decomp=np.diag(lambda_sqrt)

In [18]:
#grab the first four principal components that represent movements in the yield curve. 
#These components are typically associated with the wiggle, flex, twist, and shift of the curve.

#create a DataFrame with the first four components and scales each one by 100 to represent a basis point change.
#Each row of this DataFrame describes the sensitivity to changes in the factor at each maturity.

B = eigv_decomp @ eigenvectors.T
B = pd.DataFrame(
    data=B[:4] * 100,
    index=["wiggle", "flex", "twist", "shift"],
    columns=rate.columns
)
print(B)

         month_1   month_3   month_6    year_1    year_2    year_3    year_5  \
wiggle -2.157827 -2.235228 -2.252854 -2.165724 -1.920011 -1.754758 -1.532858   
flex    0.508147  0.381574  0.221221  0.039378 -0.180093 -0.275678 -0.314723   
twist   0.138378  0.028228 -0.071218 -0.138819 -0.166404 -0.138872 -0.020795   
shift  -0.104951  0.040896  0.071192  0.048404 -0.025235 -0.042612 -0.042125   

          year_7   year_10   year_20   year_30  
wiggle -1.399213 -1.274925 -1.185009 -1.042932  
flex   -0.300523 -0.264385 -0.247709 -0.163002  
twist   0.067139  0.138395  0.185890  0.195399  
shift  -0.024835  0.000449  0.039598  0.043038  


In [19]:
#Shock the rate curve

#Now that we have the sensitivities we can simulate changes to the yield curve. We do this by multiplying a variable by a row in B.
#For example, if we multiply the fourth row representing the shift of the yield curve by 1, 
#we would anticipate the entire yield curve to shift by the magnitude of the bottom row. 
#This means the 1 month yield would decrease by 10 basis points, the 3 month would increase by 4 basis points, 
#and so on along the curve.

#We can simulate changes in the yield curve by multiplying each row by a normally distributed variable.


In [20]:
random_shocks = np.random.normal(0, 1, size=(4,))
random_shocks @ B

month_1   -2.414728
month_3   -2.205094
month_6   -2.107770
year_1    -2.026870
year_2    -1.923964
year_3    -1.864565
year_5    -1.843155
year_7    -1.817271
year_10   -1.751934
year_20   -1.677628
year_30   -1.489913
dtype: float64